In [ ]:
model_ckpt = "meta-llama/Llama-3.2-1B"

In [ ]:
import random
import transformers
import plotly.express
import plotly.graph_objects
import matplotlib
import PIL.Image
import numpy as np
import torch
import sklearn.decomposition
from torch import Tensor

def pca(embs: Tensor, low_dim: int) -> Tensor:
    pca = sklearn.decomposition.PCA(n_components=low_dim)
    reduced_embs = pca.fit_transform(embs.detach().numpy())
    return torch.tensor(reduced_embs)

def fourier(embs: Tensor) -> Tensor:
    return torch.fft.fft(embs, dim=0)


def vis_emb(embs: Tensor, colorful: bool) -> PIL.Image.Image:
    x = embs.cpu().detach()
    x_normalized = (x - x.min()) / (x.max() - x.min())
    theme = matplotlib.colormaps["Blues"]
    if colorful:
        x_normalized = theme(x_normalized)
    else:
        x_normalized = x_normalized.numpy()
    vis = PIL.Image.fromarray((x_normalized * 255).astype(np.uint8))
    return vis

model = transformers.AutoModel.from_pretrained(model_ckpt).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckpt)

inputs_str = [
    f"{x1} + {x2}" for x1, x2 in zip(random.Random(0).sample(range(0, 1000), 1000), range(0, 1000))
]
inputs = tokenizer(inputs_str, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)


In [ ]:
for hidden_state_layer_idx in range(len(outputs.hidden_states)):
    hidden_states_last_token = outputs.hidden_states[hidden_state_layer_idx][:, -1, :]
    repr_pca = pca(hidden_states_last_token.cpu(), 64)
    repr_fft = fourier(repr_pca).abs().T.max(dim=0).values
    print(f"Layer {hidden_state_layer_idx} / {len(outputs.hidden_states)}")
    display(
        vis_emb(repr_pca.T[:16], colorful=True).resize((800, 200), PIL.Image.Resampling.NEAREST)
    )
    display(
        plotly.express.bar(
            x=torch.arange(len(repr_fft)),
            y=repr_fft.cpu().detach(),
            color_discrete_sequence=["black"],
        ).update_layout(
            showlegend=False,
            width=1200,
            height=400,
            margin=dict(l=0, r=8, t=0, b=0),
            bargap=0,
        ).update_xaxes(
            #title="Frequency",
            title=None,
            dtick=100,
            tickfont=dict(size=28),
        ).update_yaxes(
            #title=model.display_name,
            title=None,
            title_font=dict(size=32),
            tickfont=dict(size=28),
            showticklabels=False,
        ).update_traces(
            marker_line_width=0
        )
    )